# NB_hospital_daily_summary — Silver to Gold

Builds `azuredatabricks_medical.gold.hospital_daily_summary` — one row per hospital per calendar day, with test counts, abnormal/critical rates, and day-over-day volume change.

**Load type:** FULL (CREATE OR REPLACE TABLE … AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE azuredatabricks_medical.gold.hospital_daily_summary AS
WITH base AS (
    SELECT
        lr.hospital_id,
        h.hospital_name,
        lr.test_date                                                        AS summary_date,
        c.day_name,
        c.is_weekend,
        COUNT(*)                                                            AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                       AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                        AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM azuredatabricks_medical.silver.lab_results lr
    LEFT JOIN azuredatabricks_medical.silver.hospitals h
        ON lr.hospital_id = h.hospital_id
    LEFT JOIN azuredatabricks_medical.silver.calendar c
        ON c.date_key = CAST(DATE_FORMAT(lr.test_date, 'yyyyMMdd') AS INT)
    GROUP BY lr.hospital_id, h.hospital_name, lr.test_date, c.day_name, c.is_weekend
)
SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2))   AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2))   AS critical_rate_pct,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)     AS prev_day_total_tests,
    CAST(ROUND(
        (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
        / NULLIF(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date), 0) * 100
    , 2) AS DECIMAL(6,2))                                                     AS test_volume_change_pct
FROM base

In [0]:
updated_df = spark.table("azuredatabricks_medical.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))